# Vektorisierung mit NumPy

NumPy denkt in Operationen auf **ganzen Arrays** und nicht **in einzelnen Schritten**.

**Wichtig:** Vermeiden Sie, wenn immer möglich, mit `for`-Schleifen auf NumPy-Arrays zu operieren und jedes Array-Element einzeln zu besuchen. Nutzen Sie - wenn immer möglich - effiziente Vektoroperationen, die alles *auf einen Rutsch* erledigen!

In [ ]:
# notwendige Module

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# SO BITTE NICHT
xs = np.array([1., 2., 3., 4.])
ys = np.zeros_like(xs)

for i in range(xs.size):
    # Besuch jedes Array-Elements:
    ys[i] = np.sqrt(xs[i])

ys

In [ ]:
# BITTE SO:
xs = np.array([1., 2., 3., 4.])
# Erledigung der Quadratwurzel für alle Elemente in einem Rutsch
# (Vektorisierung)
ys = np.sqrt(xs)

ys

In [ ]:
# Etliche Probleme lassen sich nicht vektorisieren und müssen
# mit Schleifen erledigt werden:
xs = np.array([1., 2., 3., 4.])
ys = np.zeros_like(xs)
alpha = 0.5

ys[0] = 1.
for i in range(1, xs.size):
    # ys[i] hängt vom Ergebnis für ys[i-1] ab (rekursive Berechnung;
    # nicht vektorisierbar!).
    ys[i] = alpha * xs[i]**2 + (1 - alpha) * ys[i - 1]

ys

**Problem für Anfänger:** Wann habe ich ein Problem, das ich vektorisieren kann?

**Faustregel:** Ein Problem ist meist vektorisierbar, wenn die Berechnung für jedes Arrayelement unabhängig von den anderen erfolgt. Wenn ein Wert von vorher berechneten Ergebnissen abhängt (z. B. rekursive Formeln), ist Vektorisierung in der Regel nicht möglich.

## NumPy-Muster und Patterns für vektorisierbare Probleme

NumPy-Features, die Vektorisierung auf Arrays unterstützen:

- Elementweise Operationen auf Arrays
- Slicing
- Boolsche Masken
- Mehrdimensionalität / Dimensionsreduktion (noch nicht behandelt)
- Broadcasting (noch nicht behandelt)

### Pattern 1: Anwendung der gleichen Funktion auf jedes Arrayelement (elmentweise Operationen)

Funktionen, die direkt auf ganzen Arrays wirken können, sind so-genannte *Universal Functions* oder `ufuncs`. Diese werden unter anderem von `numpy` zur Verfügung gestellt:

In [ ]:
# Beispiel: Funktionsanwendung
x = np.linspace(0., 2. * np.pi, 10)
y = np.sin(x)

x, y

In [ ]:
# Der ufunc Typ:
type(np.sin)

### Pattern 2: Filtern und Fallunterscheidung (Boolsche Masken)

#### Beispiel 1: Filtern von Elementen


Filtere aus einem Array `x` alle Elemente mit $x > 0$:

In [ ]:
x = np.linspace(-2., 2., 10)

x_filt = x[(x > 0)]
x, x_filt

#### Beispiel 2: Verschiedene Operationen auf Teilarrays / Fallunterscheidung

Berechne für ein Array `x`:

$$
f(x) = \left\{
         \begin{array}{ll} 
           x^2 & \text{für } x < 0 \\
           \sqrt{x} & \text{für } x>= 0 \\
         \end{array}    
       \right .
$$

In [ ]:
x = np.linspace(-2., 2., 10)
y = np.zeros_like(x)

# Fallunterscheidung mit boolschen Masken:
mask = (x >= 0)

y[mask] = np.sqrt(x[mask])

# ~mask: Alles, was *nicht* von mask erfasst wird:
y[~mask] = x[~mask]**2

x, y
#plt.plot(x, y)

#### Beispiel 3: Monte-Carlo Abschätzung von $\pi$

Wir schätzen die Kreiszahl π ab, indem $N$ zufällige Punkte gleichverteilt im Einheitsquadrat $[0,1]\times[0,1]$ erzeugt werden. Der Anteil der Punkte $N_c$, die innerhalb des Vierteleinheitskreises liegen $(x^2 + y^2 \le 1)$, entspricht näherungsweise dem Flächenverhältnis $\frac{\pi}{4}$. Damit ergibt sich näherungsweise für $\pi$:

$$
\pi\approx 4\frac{N_c}{N}.
$$

<center><img src="figs/monte_carlo_pi.png" width=600 ></center>

In [ ]:
N = 1_000

xs = np.random.rand(N)
ys = np.random.rand(N)

# boolesche Maske, ob ein Pukt (x, y) innerhalb des Einheitskreises liegt:
mask = xs**2 + ys**2 < 1

# Für mathematische Operationen gilt: False = 0 und True = 1
Nc = np.sum(mask)

pi_est = 4. * Nc / N
print(pi_est)
plt.plot(xs[mask], ys[mask], 'bo', alpha=0.5)
plt.plot(xs[~mask], ys[~mask], 'ro', alpha=0.5)
plt.gca().set_aspect('equal')

### Pattern 3: Operationen zwischen Array-Nachbarelementen (Slicing)

#### Beispiel: Numerische Integration mit Riemannsumme und Trapezregel

<center><img src="figs/riemann_trapez.png" width=550 ></center>

Für eine Zerlegung des Intervalls $[a,b]$ in $n$ gleich große Teilintervalle mit Schrittweite
$$
h=\frac{b-a}{n}, \qquad x_i=a+i\,h \;\; (i=0,\dots,n)
$$
lauten die Näherungsformeln für ein numerisches Integral:

$$
\int_a^b f(x)\, dx
$$

**einfache Riemannsumme:**
$$
\int_a^b f(x)\,dx \approx \sum_{i=0}^{n-1} f\!\left(x_i\right)\,h
$$

**Trapezregel:**
$$
\int_a^b f(x)\,dx \approx \sum_{i=0}^{n-1} \frac{f(x_i)+f(x_{i+1})}{2}\,h.
$$

Wir betrachten $f(x)=\sin(x) + 0.3x + 1$ und es gilt:

$$
\int_1^6 f(x)\, dx = \frac{41}{4} + \cos(1) - \cos(6) \approx 9.830132
$$

In [ ]:
a = 1.
b = 6.
h = 1

x = np.arange(a, b + h, h)
f_x = np.sin(x) + 0.3 * x + 1.0

riemann = np.sum(f_x[:-1]) * h
riemann

In [ ]:
trapez = np.sum((f_x[1:] + f_x[:-1]) / 2.) * h
trapez

In [ ]:
riemann, trapez

## Zusammenfassung: Wann soll ich schauen, ob ein Problem vektorisierbar ist?

- gleiche Operation für alle Elemente → elementweise Arrayoperationen
- Auswahl von Daten → Masken
- Nachbarn vergleichen → Slicing
- Abhängigkeit von vorherigen Ergebnissen → **nicht** vektorisierbar